# 3. Plotly Evaluation Plots

Generate interactive evaluation plots using the **plotly** backend.
Plots are saved as HTML files for interactive exploration.

In [ ]:
import os
import subprocess
from pathlib import Path

from IPython.display import IFrame, display

os.chdir("/hpc/compgen/projects/llm_GEO_project/harmonia_metadata_agent/analysis/dstoker/harmonia")
PYTHON = ".venv/bin/python"
RESULTS_GLOB = "results/*/metrics.json"

## Step 1: Generate standard plotly plots

In [ ]:
from datetime import datetime

out_dir = f"analysis/plots_plotly_{datetime.now().strftime('%Y%m%d_%H%M')}"

cmd = [
    PYTHON, "src/evaluation/make_standard_evaluation_plots.py",
    "--metrics-glob", RESULTS_GLOB,
    "--out-dir", out_dir,
    "--backend", "plotly",
    "--figure-format", "html",
    "--backfill-row-values",
    "--verbose",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
else:
    print(f"\nPlots saved to: {out_dir}")

## Step 2: Display interactive plotly plots

In [ ]:
# Load and display plotly HTML files inline
plots_dir = Path(out_dir) / "plots"
if plots_dir.exists():
    html_files = sorted(plots_dir.glob("*.html"))
    print(f"Found {len(html_files)} interactive plots\n")
    for html_f in html_files:
        print(f"--- {html_f.name} ---")
        # IFrame works well in Jupyter for HTML plots
        display(IFrame(src=str(html_f), width="100%", height=500))
else:
    print(f"No plots directory at {plots_dir}")

## Step 3: Use visualize_metrics_cli.py for individual plot types

In [ ]:
# Example: interactive bar chart for a specific metric
cmd = [
    PYTHON, "src/evaluation/visualize_metrics_cli.py",
    "bars",
    "--metric", "avg_value_accuracy_excl_empty",
    "--metrics-glob", RESULTS_GLOB,
    "--interactive",
    "--figure-format", "html",
    "--out-dir", out_dir + "/cli_plots",
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

In [ ]:
# Example: interactive heatmap
cmd = [
    PYTHON, "src/evaluation/visualize_metrics_cli.py",
    "heatmap",
    "--metric", "accuracy_excl_empty",
    "--metrics-glob", RESULTS_GLOB,
    "--interactive",
    "--figure-format", "html",
    "--out-dir", out_dir + "/cli_plots",
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

In [ ]:
# Example: cross-model comparison
cmd = [
    PYTHON, "src/evaluation/visualize_metrics_cli.py",
    "cross-compare",
    "--metrics-glob", RESULTS_GLOB,
    "--interactive",
    "--figure-format", "html",
    "--out-dir", out_dir + "/cli_plots",
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

## Step 4: Inspect data tables

In [ ]:
import pandas as pd

tables_dir = Path(out_dir) / "tables"
if tables_dir.exists():
    for csv_f in sorted(tables_dir.glob("*.csv")):
        df = pd.read_csv(csv_f)
        print(f"\n{'='*60}")
        print(f"{csv_f.name}: {df.shape[0]} rows x {df.shape[1]} cols")
        print(f"{'='*60}")
        display(df.head(10))
else:
    print("No tables directory found")